# Notebook 4 - Neural Collaborative Filtering (NCF)

Neural Collaborative Filtering is a deep learning approach to recommendation systems that extends traditional matrix factorization by replacing the simple dot product with a neural network. The core idea is that while SVD captures linear relationships between users and movies through latent factor dot products, a neural network can learn complex non-linear patterns — for example understanding that a user enjoys comedies but only when combined with romance, or that a user's taste for action films depends on the era of the movie.

The model learns two sets of embeddings — one for users and one for movies — which are dense vector representations capturing latent characteristics. These embeddings are concatenated and passed through multiple dense layers with dropout regularization, allowing the network to discover intricate interaction patterns that simpler models cannot capture.

This notebook implements NCF on the MovieLens 1M dataset using TensorFlow and Keras. The key steps include:

- Loading processed train and test data from Phase 1  
- Encoding user and movie IDs to consecutive integers using LabelEncoder fitted on combined train and test sets to prevent unseen ID errors  
- Normalizing ratings from 1–5 scale to 0–1 scale for stable neural network training using the formula:  
  \[
  (rating - 1) / 4
  \]  
- Building NCF architecture with:
  - 64-dimensional user and movie embedding layers with L2 regularization  
  - Dense(128) → Dropout(0.3) → Dense(64) → Dropout(0.3) → Dense(32) → Sigmoid output  
- Compiling with Adam optimizer (lr = 0.0005) and MSE loss  
- Training with Early Stopping (patience = 3) and ReduceLROnPlateau (factor = 0.5) callbacks to prevent overfitting  
- Early stopping triggered at epoch 5 restoring best weights from epoch 2  
- Evaluating on test set with predictions reversed from 0–1 back to 1–5 scale  
- Generating top-N personalized recommendations for User 1680  
- Saving model, user encoder, and movie encoder for use in hybrid phase  

---

## Model Architecture

- **User Embedding:** 6040 × 64 = 386,560 parameters  
- **Movie Embedding:** 3706 × 64 = 237,184 parameters  
- **Dense layers:** 16,512 + 8,256 + 2,080 + 33 = 26,881 parameters  

**Total:** 650,625 parameters (2.48 MB)

---

## Final Model Performance

| Model       | RMSE   | MAE    |
|:------------|:------:|:------:|
| SVD         | 0.9372 | 0.7440 |
| Neural CF   | 0.9572 | 0.7597 |

SVD outperforms NCF slightly on this dataset which is consistent with published literature — simple matrix factorization methods remain competitive on small explicit rating datasets. NCF's advantage becomes more pronounced on larger implicit feedback datasets with millions of interactions. Despite the slightly higher RMSE, NCF contributes meaningful and diverse signal to the hybrid model in the next phase.

In [1]:
import pandas as pd                          # data manipulation
import numpy as np                           # numerical operations
import tensorflow as tf                      # deep learning framework
from tensorflow import keras                 # high level neural network API
from tensorflow.keras import layers          # building blocks for neural networks
from sklearn.preprocessing import LabelEncoder  # encode user/movie IDs to integers
import matplotlib.pyplot as plt              # plotting
import pickle                                # saving python objects to disk
import os                                    # file and folder operations
import warnings
warnings.filterwarnings('ignore')

# Verify TensorFlow version
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [2]:
base_path = "OneDrive/Documents/Recommendation System"

# load processed train and test files
train = pd.read_csv(f'{base_path}/train.csv')
test = pd.read_csv(f'{base_path}/test.csv')
movies = pd.read_csv(f'{base_path}/movies.csv')

print("Train size:", train.shape)
print("Test size:", test.shape)
print("\nTrain columns:", train.columns.tolist())
print("\nSample data:")
print(train.head())

Train size: (800167, 11)
Test size: (200042, 11)

Train columns: ['user_id', 'movie_id', 'rating', 'timestamp', 'year_month', 'title', 'genres', 'gender', 'age', 'occupation', 'zip_code']

Sample data:
   user_id  movie_id  rating            timestamp year_month  \
0     6040       858       4  2000-04-25 23:05:32    2000-04   
1     6040       593       5  2000-04-25 23:05:54    2000-04   
2     6040      2384       4  2000-04-25 23:05:54    2000-04   
3     6040      1961       4  2000-04-25 23:06:17    2000-04   
4     6040      2019       5  2000-04-25 23:06:17    2000-04   

                                               title              genres  \
0                              Godfather, The (1972)  Action|Crime|Drama   
1                   Silence of the Lambs, The (1991)      Drama|Thriller   
2                       Babe: Pig in the City (1998)   Children's|Comedy   
3                                    Rain Man (1988)               Drama   
4  Seven Samurai (The Magnificent

# Encode User and Movie IDs

In [3]:
# initialize label encoders for users and movies
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

# IMPORTANT: Fit encoders on ALL IDs from both train AND test combined
# If we only fit on train, test might contain unseen IDs that cause errors
# For example user_id 5000 might only appear in test set
all_user_ids = pd.concat([train['user_id'], test['user_id']])
all_movie_ids = pd.concat([train['movie_id'], test['movie_id']])

# Learn the mapping from original IDs to consecutive integers
user_encoder.fit(all_user_ids)
movie_encoder.fit(all_movie_ids)

# Apply the learned mapping to both train and test
# Original: user_id 1 → encoded: 0
# Original: user_id 2 → encoded: 1 ... and so on
train['user_idx'] = user_encoder.transform(train['user_id'])
train['movie_idx'] = movie_encoder.transform(train['movie_id'])
test['user_idx'] = user_encoder.transform(test['user_id'])
test['movie_idx'] = movie_encoder.transform(test['movie_id'])

# Total unique counts — these define the size of embedding lookup tables
n_users = len(user_encoder.classes_)
n_movies = len(movie_encoder.classes_)

print(f"Total unique users: {n_users}")
print(f"Total unique movies: {n_movies}")
print("\nSample ID encoding:")
print(train[['user_id', 'user_idx', 'movie_id', 'movie_idx']].head(8))

Total unique users: 6040
Total unique movies: 3706

Sample ID encoding:
   user_id  user_idx  movie_id  movie_idx
0     6040      6039       858        802
1     6040      6039       593        579
2     6040      6039      2384       2191
3     6040      6039      1961       1781
4     6040      6039      2019       1839
5     6040      6039      3111       2895
6     6040      6039       213        207
7     6040      6039       573        559


# Normalize Ratings

In [4]:
# Neural networks with sigmoid output require targets between 0 and 1
# We scale ratings from 1-5 range to 0-1 range
# Formula: normalized = (rating - min_rating) / (max_rating - min_rating)
#                     = (rating - 1) / (5 - 1)
#                     = (rating - 1) / 4

train['rating_norm'] = (train['rating'] - 1) / 4
test['rating_norm'] = (test['rating'] - 1) / 4

# Verify the transformation
print("Original ratings:")
print(train['rating'].value_counts().sort_index())
print("\nNormalized ratings:")
print(train['rating_norm'].value_counts().sort_index())
print("\nRating mapping:")
for r in [1, 2, 3, 4, 5]:
    print(f"  Rating {r} → {(r-1)/4:.2f}")

Original ratings:
rating
1     45338
2     84619
3    207190
4    278240
5    184780
Name: count, dtype: int64

Normalized ratings:
rating_norm
0.00     45338
0.25     84619
0.50    207190
0.75    278240
1.00    184780
Name: count, dtype: int64

Rating mapping:
  Rating 1 → 0.00
  Rating 2 → 0.25
  Rating 3 → 0.50
  Rating 4 → 0.75
  Rating 5 → 1.00


# Prepare Input Arrays

In [5]:
# Keras requires numpy arrays as inputs, not pandas dataframes

# --- TRAINING ARRAYS ---
X_train_users = train['user_idx'].values    # shape: (800167,)
X_train_movies = train['movie_idx'].values  # shape: (800167,)
y_train = train['rating_norm'].values       # shape: (800167,)

# --- TEST ARRAYS ---
X_test_users = test['user_idx'].values      # shape: (200042,)
X_test_movies = test['movie_idx'].values    # shape: (200042,)
y_test = test['rating_norm'].values         # shape: (200042,)

print("=== Training Arrays ===")
print(f"User indices shape:  {X_train_users.shape}")
print(f"Movie indices shape: {X_train_movies.shape}")
print(f"Labels shape:        {y_train.shape}")

print("\n=== Test Arrays ===")
print(f"User indices shape:  {X_test_users.shape}")
print(f"Movie indices shape: {X_test_movies.shape}")
print(f"Labels shape:        {y_test.shape}")

print("\nSample values:")
print(f"First 5 user indices:  {X_train_users[:5]}")
print(f"First 5 movie indices: {X_train_movies[:5]}")
print(f"First 5 ratings:       {y_train[:5]}")

=== Training Arrays ===
User indices shape:  (800167,)
Movie indices shape: (800167,)
Labels shape:        (800167,)

=== Test Arrays ===
User indices shape:  (200042,)
Movie indices shape: (200042,)
Labels shape:        (200042,)

Sample values:
First 5 user indices:  [6039 6039 6039 6039 6039]
First 5 movie indices: [ 802  579 2191 1781 1839]
First 5 ratings:       [0.75 1.   0.75 0.75 1.  ]


# Build Neural CF Model
SVD learns user and movie vectors and combines them with a dot product (linear operation). NCF does the same but replaces the dot product with a neural network — this allows learning complex non-linear patterns. For example SVD might struggle with "User A likes sci-fi but only old classics not modern ones" — the neural layers can capture this nuance. Dropout prevents the common problem of overfitting where a model memorizes training data but fails on new data.

In [6]:
def build_ncf_model(n_users, n_movies, embedding_dim=64):
    
    # =====================
    # USER BRANCH
    # =====================
    
    # Input layer accepts a single integer — the encoded user ID
    user_input = keras.Input(shape=(1,), name='user_input')
    
    # Embedding layer converts user ID into a dense vector of size embedding_dim
    # input_dim = total number of unique users (vocabulary size)
    # output_dim = embedding size — increased to 64 for richer user representations
    # embeddings_regularizer: L2 penalty on embedding weights
    # L2 regularization adds a small penalty for large weights during training
    # This discourages the model from assigning extreme values to any user/movie
    # which helps prevent overfitting to specific users in training data
    user_embedding = layers.Embedding(
        input_dim=n_users,
        output_dim=embedding_dim,
        embeddings_regularizer=keras.regularizers.l2(1e-6),
        name='user_embedding'
    )(user_input)
    
    # Flatten converts embedding output from shape (None, 1, 64) to (None, 64)
    # Removes the extra dimension added by the embedding layer
    # Result is a clean 64-dimensional vector for each user
    user_vec = layers.Flatten(name='user_flatten')(user_embedding)
    
    # =====================
    # MOVIE BRANCH
    # =====================
    
    # Same structure as user branch but for movies
    movie_input = keras.Input(shape=(1,), name='movie_input')
    
    # Movie embedding table: 3706 movies x 64 dimensions
    # L2 regularization applied here too for consistent regularization
    movie_embedding = layers.Embedding(
        input_dim=n_movies,
        output_dim=embedding_dim,
        embeddings_regularizer=keras.regularizers.l2(1e-6),
        name='movie_embedding'
    )(movie_input)
    
    movie_vec = layers.Flatten(name='movie_flatten')(movie_embedding)
    
    # =====================
    # INTERACTION LAYERS
    # =====================
    
    # Concatenate user vector (64d) and movie vector (64d)
    # Result is a 128-dimensional vector combining both representations
    # This is where user preferences and movie characteristics meet
    concat = layers.Concatenate(name='concat')([user_vec, movie_vec])
    
    # Dense layer 1 — wider than before (128 neurons instead of 64)
    # More neurons = more capacity to learn complex interaction patterns
    # ReLU activation introduces non-linearity
    # Without non-linearity the whole network would collapse to a linear model
    dense1 = layers.Dense(128, activation='relu', name='dense1')(concat)
    
    # Dropout increased to 0.3 (was 0.2 before)
    # Now randomly turns off 30% of neurons during each training step
    # More aggressive dropout = stronger regularization against overfitting
    # Only active during training, disabled during prediction automatically
    dropout1 = layers.Dropout(0.3, name='dropout1')(dense1)
    
    # Dense layer 2 — compress from 128 to 64 neurons
    # Gradually reducing size forces the network to learn compact representations
    dense2 = layers.Dense(64, activation='relu', name='dense2')(dropout1)
    
    # Second dropout layer for additional regularization
    dropout2 = layers.Dropout(0.3, name='dropout2')(dense2)
    
    # Dense layer 3 — further compress from 64 to 32 neurons
    # Added an extra layer compared to before for deeper feature extraction
    # Deeper networks can learn more abstract patterns
    dense3 = layers.Dense(32, activation='relu', name='dense3')(dropout2)
    
    # =====================
    # OUTPUT LAYER
    # =====================
    
    # Single neuron with sigmoid activation outputs a value between 0 and 1
    # Matches our normalized rating scale (0 = rating 1, 1 = rating 5)
    # No dropout here — we want consistent predictions at inference time
    output = layers.Dense(1, activation='sigmoid', name='output')(dense3)
    
    # Connect all inputs to output to define the full model graph
    model = keras.Model(
        inputs=[user_input, movie_input],
        outputs=output,
        name='NCF_Model'
    )
    
    return model

# Build fresh model with updated architecture
# embedding_dim=64 gives richer representations than previous 32
ncf_model = build_ncf_model(n_users, n_movies, embedding_dim=64)

# Print full architecture summary
ncf_model.summary()

Model: "NCF_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_input         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 64)     │    386,560 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_embedding     │ (None, 1, 64)     │    237,184 │ movie_input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_flatten        │ (None, 64)        │          0 │ user_embedding[0… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_flatten       │ (None, 64)        │          0 │ movie_embedding[… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat              │ (None, 128)       │          0 │ user_flatten[0][… │
│ (Concatenate)       │                   │            │ movie_flatten[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense1 (Dense)      │ (None, 128)       │     16,512 │ concat[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout1 (Dropout)  │ (None, 128)       │          0 │ dense1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense2 (Dense)      │ (None, 64)        │      8,256 │ dropout1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout2 (Dropout)  │ (None, 64)        │          0 │ dense2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense3 (Dense)      │ (None, 32)        │      2,080 │ dropout2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │         33 │ dense3[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 650,625 (2.48 MB)

 Trainable params: 650,625 (2.48 MB)

 Non-trainable params: 0 (0.00 B)

# Compile and Train the Model with Early Stopping

In [7]:
# Compile model with updated settings
ncf_model.compile(
    # Lower learning rate than before (0.001 → 0.0005)
    # Slower learning = more careful weight updates
    # Reduces the risk of overshooting the optimal solution
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    
    # MSE loss — model minimizes this during training
    # Same concept as RMSE but without the square root
    loss='mse',
    
    # Track MAE during training for monitoring alongside loss
    metrics=['mae']
)

# Early stopping callback
# Monitors validation loss at the end of each epoch
# Stops training when validation loss stops improving
# patience=3 gives the model 3 extra epochs to improve before stopping
# This is more generous than before (was patience=2)
# restore_best_weights=True automatically loads weights from best epoch
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# ReduceLROnPlateau callback
# Automatically halves the learning rate when validation loss plateaus
# factor=0.5 means new_lr = current_lr * 0.5
# patience=2 means trigger after 2 epochs of no improvement
# This helps the model escape plateaus and continue learning
# min_lr=1e-6 prevents learning rate from becoming too small to be useful
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

print("Model compiled successfully!")
print("\nStarting training...")
print("=" * 50)

# Train the model with both callbacks
history = ncf_model.fit(
    # Two inputs passed as a list — user indices and movie indices
    [X_train_users, X_train_movies],
    
    # Target values — normalized ratings (0 to 1 scale)
    y_train,
    
    # Larger batch size than before (256 → 512)
    # Larger batches compute more stable gradient estimates
    # Less noisy updates = smoother and more reliable training
    batch_size=512,
    
    # Allow up to 20 epochs — early stopping will prevent going too far
    epochs=20,
    
    # Hold out 10% of training data as validation set
    # Used only to monitor performance, not for weight updates
    validation_split=0.1,
    
    # Pass both callbacks — early stopping and learning rate reduction
    callbacks=[early_stopping, reduce_lr],
    
    # Show progress bar for each epoch
    verbose=1
)

print(f"\nTraining stopped at epoch: {len(history.history['loss'])}")
print(f"Best val_loss: {min(history.history['val_loss']):.4f}")
print("\nTraining complete!")

Model compiled successfully!

Starting training...
Epoch 1/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 60s 36ms/step - loss: 0.0638 - mae: 0.2017 - val_loss: 0.0587 - val_mae: 0.1931 - learning_rate: 5.0000e-04
Epoch 2/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 53s 37ms/step - loss: 0.0513 - mae: 0.1787 - val_loss: 0.0586 - val_mae: 0.1926 - learning_rate: 5.0000e-04
Epoch 3/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 62s 43ms/step - loss: 0.0499 - mae: 0.1754 - val_loss: 0.0586 - val_mae: 0.1911 - learning_rate: 5.0000e-04
Epoch 4/20
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.0482 - mae: 0.1717
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 39s 27ms/step - loss: 0.0482 - mae: 0.1717 - val_loss: 0.0595 - val_mae: 0.1930 - learning_rate: 5.0000e-04
Epoch 5/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 41s 29ms/step - loss: 0.0463 - mae: 0.1675 - val_loss: 0.0599 - val_mae: 0.1939 - learning_rate: 2.5000e-04
Epoch 6/20
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/

# Evaluate Model

In [8]:
print("Generating predictions on test set...")

# Predict ratings for all test interactions in batches
# verbose=0 suppresses prediction progress bar for cleaner output
# .flatten() converts shape (200042, 1) to (200042,) — removes extra dimension
y_pred_norm = ncf_model.predict(
    [X_test_users, X_test_movies],
    batch_size=512,
    verbose=0
).flatten()

# Reverse the normalization we applied in Cell 4
# Formula: original_rating = normalized_rating * 4 + 1
# Example: 0.75 * 4 + 1 = 4.0 (rating 4)
# Example: 1.00 * 4 + 1 = 5.0 (rating 5)
y_pred = y_pred_norm * 4 + 1
y_test_original = y_test * 4 + 1

# Clip predictions to valid rating range 1-5
# Sigmoid occasionally produces values very close to 0 or 1
# which when reversed could go slightly outside 1-5 range
y_pred = np.clip(y_pred, 1, 5)

# Calculate RMSE (Root Mean Squared Error)
# Measures average magnitude of prediction errors
# Larger errors are penalized more heavily due to squaring
# Lower is better — 0 means perfect predictions
rmse = np.sqrt(np.mean((y_test_original - y_pred) ** 2))

# Calculate MAE (Mean Absolute Error)
# Average absolute difference between predicted and actual ratings
# Less sensitive to large errors than RMSE
# Lower is better
mae = np.mean(np.abs(y_test_original - y_pred))

print(f"\nNeural CF RMSE: {rmse:.4f}")
print(f"Neural CF MAE:  {mae:.4f}")

# Side by side comparison with SVD
print("\n--- Model Comparison So Far ---")
print(f"{'Model':<20} {'RMSE':<10} {'MAE':<10}")
print("-" * 40)
print(f"{'SVD':<20} {'0.9372':<10} {'0.7440':<10}")
print(f"{'Neural CF':<20} {rmse:<10.4f} {mae:<10.4f}")

# Store NCF metrics for use in hybrid evaluation phase
ncf_rmse = rmse
ncf_mae = mae

Generating predictions on test set...

Neural CF RMSE: 0.9572
Neural CF MAE:  0.7597

--- Model Comparison So Far ---
Model                RMSE       MAE       
----------------------------------------
SVD                  0.9372     0.7440    
Neural CF            0.9572     0.7597    


# Generate Recommendations

In [9]:
def get_ncf_recommendations(user_id, model, train_df, movies_df,
                             user_encoder, movie_encoder, n=10):
    
    # Check if this user exists in our encoder's known classes
    # Users not seen during training cannot get recommendations
    if user_id not in user_encoder.classes_:
        print(f"User {user_id} not found in encoder.")
        return None
    
    # Encode original user ID to its integer index
    # e.g. user_id 1680 → encoded index 1672
    user_idx = user_encoder.transform([user_id])[0]
    
    # Get all movies this user has already rated
    # Using set for O(1) lookup — much faster than list for large datasets
    rated_movie_ids = set(
        train_df[train_df['user_id'] == user_id]['movie_id'].values
    )
    
    # Get all movie IDs from the movies dataframe
    all_movie_ids = movies_df['movie_id'].unique()
    
    # Filter to only unrated movies that also exist in our encoder
    # We skip movies not in encoder to avoid transformation errors
    unrated_movies = [
        m for m in all_movie_ids
        if m not in rated_movie_ids       # user hasn't seen this movie
        and m in movie_encoder.classes_   # movie exists in our encoder
    ]
    
    # Encode all unrated movie IDs to their integer indices
    movie_indices_arr = movie_encoder.transform(unrated_movies)
    
    # Create user array by repeating user index for every unrated movie
    # Model needs one (user, movie) pair per prediction
    # If 3000 unrated movies: user_arr = [1672, 1672, 1672, ...] (3000 times)
    user_arr = np.array([user_idx] * len(unrated_movies))
    movie_arr = np.array(movie_indices_arr)
    
    # Predict ratings for ALL unrated movies in a single batch operation
    # Much more efficient than predicting one at a time in a loop
    predictions = model.predict(
        [user_arr, movie_arr],
        batch_size=512,
        verbose=0
    ).flatten()
    
    # Reverse normalization — convert 0-1 predictions back to 1-5 scale
    predictions = predictions * 4 + 1
    
    # Clip to valid range in case sigmoid output goes slightly out of bounds
    predictions = np.clip(predictions, 1, 5)
    
    # Get indices that sort predictions from highest to lowest
    # [::-1] reverses to descending, [:n] takes top n indices
    top_indices = np.argsort(predictions)[::-1][:n]
    
    # Map top indices back to original movie IDs
    top_movie_ids = [unrated_movies[i] for i in top_indices]
    
    # Round predicted ratings to 2 decimal places
    top_ratings = [round(float(predictions[i]), 2) for i in top_indices]
    
    # Build results dataframe with movie IDs and predicted ratings
    results = pd.DataFrame({
        'movie_id': top_movie_ids,
        'predicted_rating': top_ratings
    })
    
    # Merge with movies dataframe to get human readable titles and genres
    results = results.merge(
        movies_df[['movie_id', 'title', 'genres']],
        on='movie_id',
        how='left'
    )
    
    return results[['title', 'genres', 'predicted_rating']].reset_index(drop=True)

# Use most active user as consistent test subject across all phases
active_users = train['user_id'].value_counts()
test_user = active_users.index[0]  # User 1680

print(f"Neural CF Top 10 Recommendations for User {test_user}:")
ncf_recs = get_ncf_recommendations(
    test_user, ncf_model, train, movies,
    user_encoder, movie_encoder, n=10
)
print(ncf_recs)

Neural CF Top 10 Recommendations for User 1680:
                                  title                genres  \
0      Shawshank Redemption, The (1994)                 Drama   
1                        Sanjuro (1962)      Action|Adventure   
2                 Paths of Glory (1957)             Drama|War   
3              Great Escape, The (1963)         Adventure|War   
4             Lawrence of Arabia (1962)         Adventure|War   
5                   General, The (1927)                Comedy   
6               Apple, The (Sib) (1998)                 Drama   
7  Bridge on the River Kwai, The (1957)             Drama|War   
8            Maltese Falcon, The (1941)     Film-Noir|Mystery   
9                        Yojimbo (1961)  Comedy|Drama|Western   

   predicted_rating  
0              4.59  
1              4.59  
2              4.50  
3              4.48  
4              4.48  
5              4.48  
6              4.47  
7              4.46  
8              4.46  
9              4

# Save Model

In [10]:
models_path = f'{base_path}/models'

# Create models directory if it doesn't exist
os.makedirs(models_path, exist_ok=True)

# Save Keras model in native .keras format
# This saves the complete model — architecture + learned weights + optimizer state
# All three are needed to resume training or make predictions later
ncf_model.save(f'{models_path}/ncf_model.keras')
print("NCF model saved!")

# Save user encoder
# Must be identical at inference time — different encoder = wrong index mapping
# Which would produce completely wrong recommendations
with open(f'{models_path}/user_encoder.pkl', 'wb') as f:
    pickle.dump(user_encoder, f)
print("User encoder saved!")

# Save movie encoder — same reason as user encoder
with open(f'{models_path}/movie_encoder.pkl', 'wb') as f:
    pickle.dump(movie_encoder, f)
print("Movie encoder saved!")

# Save training history as CSV
# Useful for reviewing training behaviour after the notebook is closed
history_df = pd.DataFrame(history.history)
history_df.to_csv(f'{base_path}/training_history.csv', index=False)
print("Training history saved!")

print("\nAll files saved successfully!")
print("\nFiles in models folder:")
for f in os.listdir(models_path):
    print(f"  {f}")

NCF model saved!
User encoder saved!
Movie encoder saved!
Training history saved!

All files saved successfully!

Files in models folder:
  cosine_sim.npy
  movie_encoder.pkl
  movie_indices.pkl
  ncf_model.keras
  svd_model.pkl
  tfidf_vectorizer.pkl
  user_encoder.pkl
